In [9]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
import pandas as pd
import random
from datetime import datetime
from pyspark.sql import SparkSession                    
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, TimestampType
import pandas as pd
import random
from datetime import datetime
import pytz

StatementMeta(, 492ebf7f-5485-409e-931e-26cdb86739e9, 11, Finished, Available, Finished)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from datetime import datetime
import random
import json

# Initialize Spark session
spark = SparkSession.builder.appName("FootTrafficSimulation").getOrCreate()

# List of Store IDs
store_ids = [f"S{str(i).zfill(3)}" for i in range(1, 11)]

# Function to generate IoTHub JSON string
def generate_iot_metadata(dt):
    formatted_time = dt.strftime('%Y-%m-%dT%H:%M:%S.%f') + "000Z"  # Adds extra 3 digits and Z
    metadata = {
        "MessageId": None,
        "CorrelationId": None,
        "ConnectionDeviceId": "retail-foottraffic-device",
        "ConnectionDeviceGenerationId": "637769785707914893",
        "EnqueuedTime": formatted_time
    }
    return json.dumps(metadata)

# Generate data
data = []
for store in store_ids:
    for _ in range(10):
        now = datetime.utcnow()
        before = random.randint(10, 30)
        after = before + random.randint(5, 20)
        iot_hub = generate_iot_metadata(now)
        data.append((now, before, after, iot_hub, store))

# Define schema
schema = StructType([
    StructField("RecordedOn", TimestampType(), True),
    StructField("before_traffic", IntegerType(), True),
    StructField("after_traffic", IntegerType(), True),
    StructField("IoTHub", StringType(), True),
    StructField("storeid", StringType(), True)
])

# Create DataFrame
df = spark.createDataFrame(data, schema)

# Display sample
df.show(10, truncate=False)

# Optional: Write to Lakehouse or table
# df.write.mode("append").format("delta").option("mergeSchema", "true").saveAsTable("FootTraffic")


StatementMeta(, 492ebf7f-5485-409e-931e-26cdb86739e9, 12, Finished, Available, Finished)

+--------------------------+--------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|RecordedOn                |before_traffic|after_traffic|IoTHub                                                                                                                                                                                               |storeid|
+--------------------------+--------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|2025-07-28 16:26:30.361383|12            |19           |{"MessageId": null, "CorrelationId": null, "ConnectionDeviceId": "retail-foottraffic-device", "ConnectionDeviceGenerationId": "637769785707914893", "En

In [11]:
!pip install azure-eventhub

StatementMeta(, 492ebf7f-5485-409e-931e-26cdb86739e9, 13, Submitted, Running, Running)

In [ ]:
import json
from azure.eventhub import EventHubProducerClient, EventData
from datetime import datetime

# Custom JSON encoder to handle datetime
class DateTimeEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, datetime):
            return obj.isoformat()  # Convert datetime to string
        return super().default(obj)

# Event Hub details

CONNECTION_STR = "#Connection string Primary Key#"
EVENT_HUB_NAME = "#Eventhub Name#"

# Initialize Event Hub producer
producer = EventHubProducerClient.from_connection_string(conn_str=CONNECTION_STR, eventhub_name=EVENT_HUB_NAME)

event_data_batch = producer.create_batch()
for row in df.collect():
    record = row.asDict()
    json_payload = json.dumps(record, cls=DateTimeEncoder)  # ✅ Handles datetime
    event_data_batch.add(EventData(json_payload))

# Send the batch
producer.send_batch(event_data_batch)
producer.close()

print("✅ Data sent to Eventstream endpoint.")

StatementMeta(, , -1, Waiting, , Waiting)